# 数据下载

下载 2022—2025 年上交所股票和上证 A 股指数数据，保存至 `data/raw/`，供 `process.ipynb` 清洗。请先在本机配置 Tushare Token 和接口权限，再按顺序执行。


In [ ]:
from pathlib import Path
import tushare as ts
import numpy as np
import pandas as pd

## 文件路径与参数

自动向上定位项目根目录，使用项目相对路径读写文件；内核工作目录应位于本项目内。下方同时定义本步骤使用的参数。


In [ ]:
PROJECT_ROOT = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / 'environment.yml').is_file() and (path / 'src').is_dir()
)

# 时间段设定
START = "20220101"
END = "20251231"

DAILY = PROJECT_ROOT / "data/raw/日频行情_2022_2025.parquet"
INDEX = PROJECT_ROOT / "data/raw/上证A股指数_2022_2025.parquet"

## 初始化数据接口

创建 Tushare 客户端，使用本机已配置的 Token。以下下载步骤会访问接口。


In [ ]:
pro = ts.pro_api()

## 1. 股票池

### 1.1 下载上交所历史股票池

合并当前上市与已退市股票，统一股票代码和上市、退市日期类型，避免股票池只包含当前仍上市的股票。


In [ ]:
pool_L = pro.stock_basic(
        exchange="SSE",
        list_status="L",
        fields=['ts_code', "name", 'list_date', 'delist_date'],
        )

pool_D = pro.stock_basic(
        exchange="SSE",
        list_status="D",
        fields=['ts_code', "name", 'list_date', 'delist_date'],
        )

pool = pd.concat([pool_L, pool_D], ignore_index=True)

pool['ts_code'] = pool['ts_code'].astype('string')
pool['name'] = pool['name'].astype('string')

for column in ["list_date", "delist_date"]:
    pool[column] = pd.to_datetime(
        pool[column],
        format="%Y%m%d",
    )

### 1.2 只保留研究区间内需要的股票

保留上市存续期与研究区间有交集的股票，供后续逐日构建股票池。


In [ ]:
pool = pool.loc[(pool["list_date"] <= END) & ((pool["delist_date"].isna())|(pool["delist_date"] >= START))].sort_values('ts_code').reset_index(drop=True)

In [ ]:
pool

## 2. 交易日历

### 2.1 下载交易日历

读取上交所交易日历，后续只保留开市日，并按日期升序下载。


In [ ]:
calendar = pro.trade_cal(
    exchange='SSE',
    start_date=START,
    end_date=END,
    fields=["cal_date", "is_open"],
    )

calendar['cal_date'] = pd.to_datetime(calendar['cal_date'], format="%Y%m%d")

### 2.2 只保留交易日

In [ ]:
calendar = calendar.loc[calendar["is_open"] == 1,['cal_date']].rename(columns = {'cal_date': 'trade_date'})
calendar = calendar.sort_values('trade_date').reset_index(drop=True)

In [ ]:
calendar.head()

## 3. 日频面板

### 3.1 下载日频数据

将交易日期转换为接口要求的格式，准备股票代码列表。


In [ ]:
trade_dates = [
    d.replace('-', '') 
    for d in calendar['trade_date'].astype('string')
    ]
ts_codes = pool['ts_code'].to_list()

### 网络请求重试

网络请求异常时采用逐步延长的等待时间重试；达到重试上限后抛出异常。


In [ ]:
import time
import random
import requests

def retry_api(func, max_retries=5, base_delay=2, **kwargs):
    """
    API 请求失败时自动重试。

    max_retries: 最多重试 5 次（首次请求不计入）
    base_delay: 初始等待 2 秒
    """
    for attempt in range(max_retries + 1):
        try:
            return func(**kwargs)

        except requests.exceptions.RequestException as e:
            # 最后一次尝试也失败，则抛出异常
            if attempt == max_retries:
                raise

            # 指数退避：2、4、8、16、32 秒，并加入随机等待
            wait = min(base_delay * (2 ** attempt), 60)
            wait += random.uniform(0, 1)

            print(
                f"{func.__name__} 请求失败：{e}\n"
                f"第 {attempt + 1}/{max_retries} 次重试，"
                f"{wait:.1f} 秒后重新请求..."
            )

            time.sleep(wait)

### 按交易日下载

逐日读取行情、每日指标、复权因子、停牌及 ST 记录。当日五类请求全部成功后才加入列表，失败日期记录在 `failed_dates` 中。


In [ ]:
# 下载前初始化
day_list = []
day_basic_list = []
day_adj_list = []
day_suspend_list = []
day_st_list = []

failed_dates = []

for date in trade_dates:
    try:
        # 1. 日线行情
        day = retry_api(
            pro.daily,
            trade_date=date,
            fields = [
                'ts_code',       # 股票代码
                'trade_date',    # 交易日期
                'open',          # 开盘价
                'close',         # 收盘价
                'low',           # 最低价
                'high',          # 最高价
                'vol',           # 成交量（手）
                'amount',        # 成交额（千元）
                ]
        )

        # 2. 每日指标
        day_basic = retry_api(
            pro.daily_basic,
            trade_date=date,
           fields=[
                'ts_code',
                'trade_date',
                "turnover_rate",   # 换手率: 成交量/流通股本
                "total_share",     # 总股本（万股）
                "float_share",     # 流通股本（万股）
                "free_share",      # 自由流通股本（万股）
                "total_mv",        # 总市值（万元）
                "circ_mv",         # 流通市值（万元）
                "pe",              # 市盈率: 总市值/最近一期年报净利润, 亏损的PE为空
                "pe_ttm",          # 滚动市盈率: 总市值/滚动计算最近4个季度的净利润, 亏损的PE为空
                "pb",              # 市净率: 总市值/(净资产-其他权益工具)
                "ps",              # 市销率: 总市值/最近一期年报收入
                "ps_ttm",          # 滚动市销率: 总市值/滚动计算最近4个季度的净利润
                "limit_status"     # 收盘涨跌状态: 0-平盘，1-上涨(不含涨停)，2-涨停(不含一字涨停)，3-一字涨停，4-下跌(不含跌停)，5-跌停(不含一字跌停)，6-一字跌停
                ]
        )

        # 3. 复权因子
        day_adj = retry_api(
            pro.adj_factor,
            trade_date=date,
            fields=[
                "ts_code",
                "trade_date",
                "adj_factor"   # 复权因子
                ]
        )

        # 4. 停复牌信息
        day_suspend = retry_api(
            pro.suspend_d,
            suspend_type='S',
            trade_date=date
        )

        # 5. 特殊处理信息
        day_st = retry_api(
                    pro.stock_st,
                    trade_date=date,
                    fields = [
                        'ts_code',       # 股票代码
                        'trade_date',    # 交易日期
                        "type",          # 特殊处理类型
                        ]
                )

    except Exception as e:
        # 某个 API 多次重试后仍失败，记录日期并继续
        failed_dates.append(date)
        print(f'{date} 下载失败，已记录：{e}')
        continue

    # 只有当天所有 API 都成功后，才统一追加
    day_list.append(day)
    day_basic_list.append(day_basic)
    day_adj_list.append(day_adj)
    day_suspend_list.append(day_suspend)
    day_st_list.append(day_st)

    print(f'{date} 下载完成')

print(f'\n下载结束，共有 {len(failed_dates)} 个日期失败：')
print(failed_dates)

### 汇总下载结果

分别拼接各类日频记录并排序，作为下一步面板合并的输入。


In [ ]:
daily = pd.concat(day_list, ignore_index=True).sort_values(["trade_date", "ts_code"]).reset_index(drop=True)
daily_basic = pd.concat(day_basic_list, ignore_index=True).sort_values(["trade_date", "ts_code"]).reset_index(drop=True)
daily_adj = pd.concat(day_adj_list, ignore_index=True).sort_values(["trade_date", "ts_code"]).reset_index(drop=True)
daily_suspend = pd.concat(day_suspend_list, ignore_index=True).sort_values(["trade_date", "ts_code"]).reset_index(drop=True)
daily_st = pd.concat(day_st_list, ignore_index=True).sort_values(["trade_date", "ts_code"]).reset_index(drop=True)

### 3.2 合并日频数据

按交易日和股票上市存续期构建面板骨架，再合并各类数据，使停牌股票仍保留市场交易日位置。


In [ ]:
pool_days = []
for date in calendar['trade_date']:
    pool_day = pool.loc[(pool["list_date"] <= date) & ((pool["delist_date"].isna())|(pool["delist_date"] > date)),['ts_code']]
    pool_day.insert(loc=0, column="trade_date", value=date)
    pool_days.append(pool_day)

panel = pd.concat(pool_days, ignore_index=True).sort_values(["trade_date", "ts_code"]).reset_index(drop=True)

### 去重与均价

各类数据按日期和股票去重，统一 ST 类型列名。按 `amount * 10 / vol` 计算日均成交价，换算成交额与成交量单位。


In [ ]:
daily = daily.drop_duplicates(subset=['trade_date', 'ts_code'])
daily_basic = daily_basic.drop_duplicates(subset=['trade_date', 'ts_code'])
daily_adj = daily_adj.drop_duplicates(subset=['trade_date', 'ts_code'])
daily_suspend = daily_suspend.drop_duplicates(subset=['trade_date', 'ts_code'])
daily_st = daily_st.drop_duplicates(subset=['ts_code', "trade_date"]).rename(columns={"type": "ST_type"})
daily["vwap"] = (daily["amount"] * 10 / daily["vol"]).round(4)

### 对齐面板

统一键字段类型，再按 `trade_date + ts_code` 左连接至面板；未取得的字段保留缺失，交由清洗步骤处理。


In [ ]:
for day in [daily, daily_basic, daily_adj, daily_suspend, daily_st]:
    day['ts_code'] = day['ts_code'].astype('string')
    day['trade_date'] = pd.to_datetime(day['trade_date'], format="%Y%m%d")
    panel = pd.merge(panel, day, on=['trade_date', 'ts_code'] ,how = 'left')

### 保存股票原始面板

创建原始数据目录并保存至 `DAILY`。这是已合并的原始字段面板，尚未完成状态清洗、后复权和收益率计算。


In [ ]:
DAILY.parent.mkdir(parents=True, exist_ok=True)
panel.to_parquet(DAILY, index=False)

## 4. 上证指数行情

### 4.1 下载上证指数日行情

读取 `000002.SH` 上证 A 股指数，作为风格 Beta、部分 Alpha 和回测基准的数据来源。


In [ ]:
index = pro.index_daily(
    ts_code="000002.SH",
    start_date=START,
    end_date=END,
    fields=
    [
        "trade_date",
        "open",
        "close",
        "high",
        "low",
        "vol",
        "amount"
    ]
)

In [ ]:
index['trade_date'] = pd.to_datetime(index['trade_date'], format="%Y%m%d")
index = index.sort_values("trade_date").reset_index(drop=True)

### 保存指数原始行情

日期转换并排序后，保存至 `INDEX`。下一步运行 `process.ipynb`。


In [ ]:
index.to_parquet(INDEX, index=False)